In [1]:
import dlt
import duckdb
import pandas as pd

In [2]:
pipeline = dlt.attach(pipeline_name="taxi_pipeline")

with pipeline.sql_client() as client:
    with client.execute_query("SELECT COUNT(*) FROM taxi_data.rides") as cur:
        print("Row count:", cur.fetchone()[0])

Row count: 10000


In [4]:
con = duckdb.connect("taxi_pipeline.duckdb", read_only=True)
df = con.execute("SELECT * FROM taxi_data.rides LIMIT 5").df()
con.close()
df

,end_lat,end_lon,fare_amt,passenger_count,payment_type,start_lat,start_lon,tip_amt,tolls_amt,total_amt,trip_distance,trip_dropoff_date_time,trip_pickup_date_time,surcharge,vendor_name,_dlt_load_id,_dlt_id,store_and_forward
0,40.742963,-73.980072,45.0,1,Credit,40.641525,-73.787442,9.0,4.15,58.15,17.52,2009-06-14 18:48:00-05:00,2009-06-14 18:23:00-05:00,0.0,VTS,1772471114.0825934,vfVObv2B10rA/w,NaN
1,40.740187,-74.005698,6.5,1,Credit,40.722065,-74.009767,1.0,0.00,8.50,1.56,2009-06-18 12:43:00-05:00,2009-06-18 12:35:00-05:00,1.0,VTS,1772471114.0825934,EeA1/O3s1kuTpg,NaN
2,40.718043,-74.004745,12.5,5,Credit,40.761945,-73.983038,2.0,0.00,15.50,3.37,2009-06-10 13:27:00-05:00,2009-06-10 13:08:00-05:00,1.0,VTS,1772471114.0825934,0JK4/0XaupbnkA,NaN
3,40.739637,-73.985233,4.9,1,CASH,40.749802,-73.992247,0.0,0.00,5.40,1.11,2009-06-14 18:58:00-05:00,2009-06-14 18:54:00-05:00,0.5,VTS,1772471114.0825934,LboxqYuskXKWzw,NaN
4,40.730032,-73.852693,25.7,1,CASH,40.776825,-73.949233,0.0,4.15,29.85,11.09,2009-06-13 08:23:00-05:00,2009-06-13 08:01:00-05:00,0.0,VTS,1772471114.0825934,cl33EsJr6eNJmQ,NaN


In [3]:
con = duckdb.connect("taxi_pipeline.duckdb", read_only=True)
df = con.execute("""
    SELECT
        MIN(trip_pickup_date_time) AS start_date,
        MAX(trip_dropoff_date_time) AS end_date
    FROM taxi_data.rides
""").df()
con.close()
df

,start_date,end_date
0,2009-06-01 06:33:00-05:00,2009-06-30 19:03:00-05:00


In [4]:
con = duckdb.connect("taxi_pipeline.duckdb", read_only=True)
df = con.execute("""
    SELECT
        payment_type,
        COUNT(*) AS trips,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS proportion
    FROM taxi_data.rides
    GROUP BY payment_type
    ORDER BY trips DESC
""").df()
con.close()
df

,payment_type,trips,proportion
0,CASH,7235,72.35
1,Credit,2666,26.66
2,Cash,97,0.97
3,No Charge,1,0.01
4,Dispute,1,0.01


In [6]:
con = duckdb.connect("taxi_pipeline.duckdb", read_only=True)
df = con.execute("""
    SELECT
        SUM(tip_amt) AS total_amount_tips
    FROM taxi_data.rides
""").df()
con.close()
df

,total_amount_tips
0,6063.41
